In [1]:
# =============================================================================
# STAGE 9 — Deployment (Model Saving & Streamlit Prep)
# -----------------------------------------------------------------------------
# Project: Student Performance Prediction & Early Intervention (#23)
# Team   : Abhi | Aparna | Jia  |  Predictive Analytics 2025-26
# Run    : python stage9_deployment.py   (after stage8)
# Then   : cd ../streamlit_app && streamlit run app.py
# =============================================================================

import os
import warnings
import joblib
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

MODELS_DIR = "../models"
APP_DIR    = "../streamlit_app"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(APP_DIR, exist_ok=True)


# ── 9.1  Verify all model artifacts exist ────────────────────────────────────
def verify_artifacts():
    print("=" * 55)
    print("  9.1  VERIFY MODEL ARTIFACTS")
    print("=" * 55)
    files = {
        f"{MODELS_DIR}/gradient_boosting_model.pkl": "Best model (GB)",
        f"{MODELS_DIR}/trained_models.pkl"         : "All 3 models",
        f"{MODELS_DIR}/scaler.pkl"                 : "StandardScaler",
        f"{MODELS_DIR}/selected_features.pkl"      : "Feature list",
    }
    all_ok = True
    for path, desc in files.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"  {status}  {desc:<30} {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise RuntimeError(
            "Some artifacts are missing.\n"
            "Please run stage6_model_building.py first."
        )
    print("\n  All artifacts present.\n")


# ── 9.2  Quick smoke-test: run a prediction ───────────────────────────────────
def smoke_test():
    print("=" * 55)
    print("  9.2  SMOKE TEST — SAMPLE PREDICTION")
    print("=" * 55)

    model             = joblib.load(f"{MODELS_DIR}/gradient_boosting_model.pkl")
    scaler            = joblib.load(f"{MODELS_DIR}/scaler.pkl")
    selected_features = joblib.load(f"{MODELS_DIR}/selected_features.pkl")

    print(f"  Features expected by model: {selected_features}")

    # Build a sample student matching the feature list
    sample = {f: 0 for f in selected_features}
    # Fill in sensible values for common features
    overrides = {
        "G1": 12, "G2": 13, "avg_grade": 12.5,
        "failures": 0, "studytime": 2, "absences": 4,
        "higher": 1, "Medu": 3, "avg_parent_edu": 3.0,
        "alcohol_total": 2, "social_index": 2,
    }
    for k, v in overrides.items():
        if k in sample:
            sample[k] = v

    X_sample = np.array([[sample[f] for f in selected_features]])
    X_scaled = scaler.transform(X_sample)

    pred  = model.predict(X_scaled)[0]
    proba = model.predict_proba(X_scaled)[0]

    print(f"\n  Sample student input:")
    for k, v in sample.items():
        print(f"    {k:<20} = {v}")
    print(f"\n  Prediction : {'PASS ✅' if pred == 1 else 'FAIL ⚠️'}")
    print(f"  Probability: Fail={proba[0]:.3f}  Pass={proba[1]:.3f}")
    print("\n  ✅  Smoke test passed — model loads and predicts correctly.")


# ── 9.3  Print Streamlit deployment instructions ──────────────────────────────
def print_deployment_instructions():
    print("\n" + "=" * 55)
    print("  9.3  HOW TO DEPLOY THE STREAMLIT APP")
    print("=" * 55)
    print("""
  LOCAL DEPLOYMENT
  ─────────────────
  1.  Install streamlit:
        pip install streamlit

  2.  Run the app:
        cd ../streamlit_app
        streamlit run app.py

  3.  Open browser at:
        http://localhost:8501

  ──────────────────────────────────────────────────────
  CLOUD DEPLOYMENT (Streamlit Community Cloud — FREE)
  ─────────────────
  1.  Push your project to a PUBLIC GitHub repository.

  2.  Go to https://share.streamlit.io
      Sign in with your GitHub account.

  3.  Click "New app" → select your repo.
      Set:
        Branch     : main
        Main file  : streamlit_app/app.py

  4.  Click "Deploy" — takes ~2 minutes.

  5.  Copy the live URL (e.g. https://your-app.streamlit.app)
      and paste it into:
        • README.md
        • The PPT presentation
        • Submission portal

  ──────────────────────────────────────────────────────
  CHECKLIST BEFORE DEPLOYMENT
  ─────────────────
  □  requirements.txt includes: streamlit, scikit-learn,
     imbalanced-learn, lime, joblib, pandas, numpy,
     matplotlib, seaborn
  □  ../models/ folder is committed to GitHub
  □  app.py loads model from relative path ../models/
  □  Deployment link is working and shareable
  □  README contains the live URL + screenshot
""")


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    verify_artifacts()
    smoke_test()
    print_deployment_instructions()

    print("✅  Stage 9 complete.")
    print("    Next  : python stage10_summary.py  (Abhi)")


  9.1  VERIFY MODEL ARTIFACTS
  ✅  Best model (GB)                ../models/gradient_boosting_model.pkl
  ✅  All 3 models                   ../models/trained_models.pkl
  ✅  StandardScaler                 ../models/scaler.pkl
  ✅  Feature list                   ../models/selected_features.pkl

  All artifacts present.

  9.2  SMOKE TEST — SAMPLE PREDICTION
  Features expected by model: ['avg_grade', 'G2', 'Fedu', 'paid', 'absences', 'G1', 'Fjob', 'famsize', 'Dalc', 'reason']

  Sample student input:
    avg_grade            = 12.5
    G2                   = 13
    Fedu                 = 0
    paid                 = 0
    absences             = 4
    G1                   = 12
    Fjob                 = 0
    famsize              = 0
    Dalc                 = 0
    reason               = 0

  Prediction : FAIL ⚠️
  Probability: Fail=0.752  Pass=0.248

  ✅  Smoke test passed — model loads and predicts correctly.

  9.3  HOW TO DEPLOY THE STREAMLIT APP

  LOCAL DEPLOYMENT
  ──────────────